In [ ]:
# =========================================================
# 응급원무팀 3교대 근무표 생성 (OR-Tools CP-SAT)
# - 9명(A~I) 전체를 변수로 풀되,
# - A(최영철)는 N 금지(하드) + "E-휴무-D" 패턴을 최대한 유지(소프트)
# - 비A(B~I)는 기존 규칙(나이트/연속/특수조건/공정성/휴무최소 등) 적용
#
# 설치:
#   pip install ortools holidayskr
# =========================================================

from ortools.sat.python import cp_model
from datetime import date, timedelta
import calendar
from holidayskr import year_holidays

# =========================================================
# 0) 이름 맵핑
# =========================================================
NAME_MAP = {
    "A": "최영철",
    "B": "홍진우",
    "C": "김다영",
    "D": "강승민",
    "E": "문승환",
    "F": "라영일",
    "G": "이병욱",
    "H": "김동명",
    "I": "김선우",
}

A_ID = "A"
STAFF_IDS = ["B", "C", "D", "E", "F", "G", "H", "I"]  # 비A 8명(규칙 대상)
ALL_IDS = [A_ID] + STAFF_IDS

# =========================================================
# 1) 입력 영역
# =========================================================
YEAR = 2026
MONTH = 3

# 전체 기준 일일 필요 인원
REQ_D_TOTAL = 2
REQ_E_TOTAL = 2

# 전체 기준 N 필요 인원(1~2). 비우면 자동 생성 + 합계 48로 보정
N_REQ_TOTAL = []

# 빨간날 추가 지정(선택)
EXTRA_RED_DATES = [
    # date(2026, 3, 15),
]

# 휴무 하드 최소: "빨간날 개수" 또는 "빨간날-1" 중 선택
# 요청: 10일이면 10 또는 9 유지 -> 최소는 9(= MIN_OFF-1), 목표는 MIN_OFF
USE_MIN_OFF_MINUS_1 = True

# =========================================================
# 2) 달력/유틸
# =========================================================
def days_in_month(year: int, month: int) -> int:
    return calendar.monthrange(year, month)[1]

def weekday_kor(dt: date) -> str:
    kor = ["월", "화", "수", "목", "금", "토", "일"]
    return kor[dt.weekday()]

def is_sunday(dt: date) -> bool:
    return dt.weekday() == 6

def is_weekend(dt: date) -> bool:
    return dt.weekday() in (5, 6)

NUM_DAYS = days_in_month(YEAR, MONTH)
START_DATE = date(YEAR, MONTH, 1)
days = range(NUM_DAYS)

# =========================================================
# 2-1) 빨간날(주말 + 공휴일/대체공휴일) 계산
# =========================================================
def get_kr_holidays_in_month(year: int, month: int) -> dict[date, str]:
    out: dict[date, str] = {}
    for dt, name in year_holidays(str(year)):  # dt: datetime.date
        if dt.year == year and dt.month == month:
            out[dt] = name
    return out

def calc_red_dates(year: int, month: int, extra_red_dates: list[date]) -> tuple[set[date], dict[date, str]]:
    last = calendar.monthrange(year, month)[1]
    weekend_set = {date(year, month, d) for d in range(1, last + 1) if is_weekend(date(year, month, d))}
    holiday_name_map = get_kr_holidays_in_month(year, month)  # 대체공휴일 포함
    holiday_set = set(holiday_name_map.keys())
    extra_set = {d for d in extra_red_dates if d.year == year and d.month == month}
    red_dates = weekend_set | holiday_set | extra_set
    return red_dates, holiday_name_map

RED_DATES, HOLIDAY_NAME_MAP = calc_red_dates(YEAR, MONTH, EXTRA_RED_DATES)
MIN_OFF = len(RED_DATES)
MIN_OFF_HARD = max(0, MIN_OFF - 1) if USE_MIN_OFF_MINUS_1 else MIN_OFF
OFF_TARGET = MIN_OFF  # 목표는 빨간날 개수에 맞추기

print(f"== 빨간날(주말+공휴일+대체공휴일) ==  총 {MIN_OFF}일")
print(f"== 휴무 기준 ==  하드 최소 {MIN_OFF_HARD}일, 목표 {OFF_TARGET}일")
print("== 빨간날 목록 ==")
for dt in sorted(RED_DATES):
    tag = HOLIDAY_NAME_MAP.get(dt, "주말/추가")
    print(dt.isoformat(), tag)

# =========================================================
# 2-2) N_REQ_TOTAL 기본 생성 + 합계 48로 보정
#   - 비A 8명에게 N=6(하드) => 총 N 공급량 = 48
#   - A는 N 금지(하드)
#   - 따라서 전체 N 총량도 48이어야 해가 존재 가능
# =========================================================
TARGET_TOTAL_N = 8 * 6  # 48

if not N_REQ_TOTAL:
    tmp = []
    for d in range(NUM_DAYS):
        dt = START_DATE + timedelta(days=d)
        tmp.append(1 if is_weekend(dt) else 2)  # 기본: 평일2/주말1
    N_REQ_TOTAL = tmp

cur_total_n = sum(N_REQ_TOTAL)
if cur_total_n > TARGET_TOTAL_N:
    candidates = [d for d in range(NUM_DAYS)
                  if (not is_weekend(START_DATE + timedelta(days=d))) and N_REQ_TOTAL[d] == 2]
    need = cur_total_n - TARGET_TOTAL_N
    if need > len(candidates):
        raise SystemExit("[불가능] N 총량을 48로 낮출 후보(평일 2->1)가 부족합니다.")
    for d in candidates[:need]:
        N_REQ_TOTAL[d] = 1
elif cur_total_n < TARGET_TOTAL_N:
    candidates = [d for d in range(NUM_DAYS)
                  if (not is_weekend(START_DATE + timedelta(days=d))) and N_REQ_TOTAL[d] == 1]
    need = TARGET_TOTAL_N - cur_total_n
    if need > len(candidates):
        raise SystemExit("[불가능] N 총량을 48로 올릴 후보(평일 1->2)가 부족합니다.")
    for d in candidates[:need]:
        N_REQ_TOTAL[d] = 2

assert sum(N_REQ_TOTAL) == TARGET_TOTAL_N, (sum(N_REQ_TOTAL), TARGET_TOTAL_N)
print("\n== N 총량 체크 ==")
print("sum(N_REQ_TOTAL) =", sum(N_REQ_TOTAL), "(목표 48)")
print("N(1) 일수:", sum(1 for v in N_REQ_TOTAL if v == 1), "/ N(2) 일수:", sum(1 for v in N_REQ_TOTAL if v == 2))

# =========================================================
# 3) 모델 구성 (9명 전체 변수)
# =========================================================
SHIFT_D, SHIFT_E, SHIFT_N, SHIFT_OFF = 0, 1, 2, 3
SHIFT_NAMES = {SHIFT_D: "D", SHIFT_E: "E", SHIFT_N: "N", SHIFT_OFF: "-"}

model = cp_model.CpModel()

P_ALL = len(ALL_IDS)  # 9
idxA = ALL_IDS.index("A")
idxB = ALL_IDS.index("B")
idxC = ALL_IDS.index("C")

people_all = range(P_ALL)
people_nonA = range(P_ALL)  # 임시, 아래에서 제외 처리
people_nonA = [i for i in people_all if i != idxA]

shifts = (SHIFT_D, SHIFT_E, SHIFT_N, SHIFT_OFF)

x = {(i, d, s): model.NewBoolVar(f"x_{ALL_IDS[i]}_{d}_{s}")
     for i in people_all for d in days for s in shifts}

# 하루 1인 1개
for i in people_all:
    for d in days:
        model.Add(sum(x[i, d, s] for s in shifts) == 1)

# =========================================================
# 4) 일일 인원 충족(전체 기준)
# =========================================================
for d in days:
    model.Add(sum(x[i, d, SHIFT_D] for i in people_all) == REQ_D_TOTAL)
    model.Add(sum(x[i, d, SHIFT_E] for i in people_all) == REQ_E_TOTAL)
    model.Add(sum(x[i, d, SHIFT_N] for i in people_all) == N_REQ_TOTAL[d])

# =========================================================
# 5) 하드 규칙
# =========================================================
# ---- A(최영철) ----
# A는 N 금지(하드)
for d in days:
    model.Add(x[idxA, d, SHIFT_N] == 0)

# ---- 비A(B~I) ----
# 비A: N 정확히 6회(하드)
for i in people_nonA:
    model.Add(sum(x[i, d, SHIFT_N] for d in days) == 6)

# 비A: N 최대 2연속
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_N] + x[i, d+2, SHIFT_N] <= 2)

# 비A: N 다음날 D/E 금지
for i in people_nonA:
    for d in range(NUM_DAYS - 1):
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_D] <= 1)
        model.Add(x[i, d, SHIFT_N] + x[i, d+1, SHIFT_E] <= 1)

# 비A: N 2일 후 D 금지
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        model.Add(x[i, d, SHIFT_N] + x[i, d+2, SHIFT_D] <= 1)

# (비A만) E 다음날 D 금지  ※A는 배려(깨도 됨)
for i in people_nonA:
    for d in range(NUM_DAYS - 1):
        model.Add(x[i, d, SHIFT_E] + x[i, d+1, SHIFT_D] <= 1)

# B/C 특별조건
for d in days:
    if N_REQ_TOTAL[d] == 1:
        model.Add(x[idxB, d, SHIFT_N] == 0)
        model.Add(x[idxC, d, SHIFT_N] == 0)
    else:
        model.Add(x[idxB, d, SHIFT_N] + x[idxC, d, SHIFT_N] <= 1)

# =========================================================
# 6) 휴무/근무 카운트 + 하드 최소 휴무
# =========================================================
OFF_cnt = {}
WORK_cnt = {}
work = {}

for i in people_all:
    OFF_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"OFFcnt_{ALL_IDS[i]}")
    model.Add(OFF_cnt[i] == sum(x[i, d, SHIFT_OFF] for d in days))

    work[i] = {}
    for d in days:
        work[i][d] = model.NewBoolVar(f"work_{ALL_IDS[i]}_{d}")
        model.Add(work[i][d] == 1 - x[i, d, SHIFT_OFF])

    WORK_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"WORKcnt_{ALL_IDS[i]}")
    model.Add(WORK_cnt[i] == sum(work[i][d] for d in days))

    # ✅ 하드: 최소 휴무(빨간날 or 빨간날-1)
    model.Add(OFF_cnt[i] >= MIN_OFF_HARD)

# =========================================================
# 7) 연속근무 제한(전원): 6연속 금지(하드) / 5연속 지양(소프트)
# =========================================================
for i in people_all:
    for d in range(NUM_DAYS - 5):
        model.Add(sum(work[i][d+k] for k in range(6)) <= 5)

five_consec_flags = []
for i in people_all:
    for d in range(NUM_DAYS - 4):
        f = model.NewBoolVar(f"five_consec_{ALL_IDS[i]}_{d}")
        model.Add(sum(work[i][d+k] for k in range(5)) == 5).OnlyEnforceIf(f)
        model.Add(sum(work[i][d+k] for k in range(5)) <= 4).OnlyEnforceIf(f.Not())
        five_consec_flags.append(f)

# =========================================================
# 8) 소프트 규칙(최적화 항목)
# =========================================================
# (소프트) 휴무 목표 OFF_TARGET(=빨간날 개수)에 가깝게
OFF_dev = {}
for i in people_all:
    OFF_dev[i] = model.NewIntVar(0, NUM_DAYS, f"OFFdev_{ALL_IDS[i]}")
    diff = model.NewIntVar(-NUM_DAYS, NUM_DAYS, f"OFFdiff_{ALL_IDS[i]}")
    model.Add(diff == OFF_cnt[i] - OFF_TARGET)
    model.AddAbsEquality(OFF_dev[i], diff)

# (소프트) 비A의 D/E 균일화
D_cnt, E_cnt = {}, {}
for i in people_nonA:
    D_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"Dcnt_{ALL_IDS[i]}")
    E_cnt[i] = model.NewIntVar(0, NUM_DAYS, f"Ecnt_{ALL_IDS[i]}")
    model.Add(D_cnt[i] == sum(x[i, d, SHIFT_D] for d in days))
    model.Add(E_cnt[i] == sum(x[i, d, SHIFT_E] for d in days))

DE_gap = {}
for i in people_nonA:
    DE_gap[i] = model.NewIntVar(0, NUM_DAYS, f"DEgap_{ALL_IDS[i]}")
    diff = model.NewIntVar(-NUM_DAYS, NUM_DAYS, f"DEdiff_{ALL_IDS[i]}")
    model.Add(diff == D_cnt[i] - E_cnt[i])
    model.AddAbsEquality(DE_gap[i], diff)

maxD = model.NewIntVar(0, NUM_DAYS, "maxD")
minD = model.NewIntVar(0, NUM_DAYS, "minD")
maxE = model.NewIntVar(0, NUM_DAYS, "maxE")
minE = model.NewIntVar(0, NUM_DAYS, "minE")
model.AddMaxEquality(maxD, list(D_cnt.values()))
model.AddMinEquality(minD, list(D_cnt.values()))
model.AddMaxEquality(maxE, list(E_cnt.values()))
model.AddMinEquality(minE, list(E_cnt.values()))

# (소프트) 휴무 3연속 지양(비A)
triple_off_flags = []
for i in people_nonA:
    for d in range(NUM_DAYS - 2):
        t = model.NewBoolVar(f"tripleOFF_{ALL_IDS[i]}_{d}")
        model.Add(x[i, d, SHIFT_OFF] + x[i, d+1, SHIFT_OFF] + x[i, d+2, SHIFT_OFF] == 3).OnlyEnforceIf(t)
        model.Add(x[i, d, SHIFT_OFF] + x[i, d+1, SHIFT_OFF] + x[i, d+2, SHIFT_OFF] <= 2).OnlyEnforceIf(t.Not())
        triple_off_flags.append(t)

# (소프트) 일요일 D 월 1회 이상(비A)
sunday_indexes = [d for d in days if is_sunday(START_DATE + timedelta(days=d))]
missing_sunD = {}
for i in people_nonA:
    m = model.NewBoolVar(f"missSunD_{ALL_IDS[i]}")
    sum_sunD = sum(x[i, d, SHIFT_D] for d in sunday_indexes)
    model.Add(sum_sunD >= 1).OnlyEnforceIf(m.Not())
    model.Add(sum_sunD == 0).OnlyEnforceIf(m)
    missing_sunD[i] = m

# (소프트) 빨간날 근무 편중 최소화(비A)
special_days = sorted([(dt - START_DATE).days for dt in RED_DATES])
special_work = {}
for i in people_nonA:
    special_work[i] = model.NewIntVar(0, len(special_days), f"SPW_{ALL_IDS[i]}")
    model.Add(special_work[i] == sum(
        x[i, d, SHIFT_D] + x[i, d, SHIFT_E] + x[i, d, SHIFT_N]
        for d in special_days
    ))
maxSP = model.NewIntVar(0, len(special_days), "maxSP")
minSP = model.NewIntVar(0, len(special_days), "minSP")
model.AddMaxEquality(maxSP, list(special_work.values()))
model.AddMinEquality(minSP, list(special_work.values()))

# =========================================================
# 9) A(최영철) 배려: 패턴 "E -> OFF -> D" 최대한 유지(소프트)
#    - 완벽 고정 X, 어기면 페널티만
# =========================================================
A_pattern_miss = []
for d in days:
    r = d % 3
    if r == 0:
        pref = x[idxA, d, SHIFT_E]
    elif r == 1:
        pref = x[idxA, d, SHIFT_OFF]
    else:
        pref = x[idxA, d, SHIFT_D]

    miss = model.NewBoolVar(f"A_miss_{d}")
    model.Add(miss + pref == 1)  # miss = 1 - pref
    A_pattern_miss.append(miss)

# 1일은 가능하면 E(강한 소프트)
A_day1_notE = model.NewBoolVar("A_day1_notE")
model.Add(A_day1_notE + x[idxA, 0, SHIFT_E] == 1)

# =========================================================
# 10) 목적함수(가중치)
# =========================================================
W_OFF_TARGET = 250
W_A_PATTERN_MISS = 80
W_A_DAY1_E = 200

W_FIVE_CONSEC = 400
W_TRIPLE_OFF = 200
W_MISS_SUNDAY_D = 500

W_DE_GAP = 200
W_FAIR_D = 120
W_FAIR_E = 120
W_SPECIAL_FAIR = 20

model.Minimize(
    W_OFF_TARGET * sum(OFF_dev.values())
    + W_A_PATTERN_MISS * sum(A_pattern_miss)
    + W_A_DAY1_E * A_day1_notE
    + W_FIVE_CONSEC * sum(five_consec_flags)
    + W_TRIPLE_OFF * sum(triple_off_flags)
    + W_MISS_SUNDAY_D * sum(missing_sunD.values())
    + W_DE_GAP * sum(DE_gap.values())
    + W_FAIR_D * (maxD - minD)
    + W_FAIR_E * (maxE - minE)
    + W_SPECIAL_FAIR * (maxSP - minSP)
)

# =========================================================
# 11) Solve
# =========================================================
solver = cp_model.CpSolver()
solver.parameters.max_time_in_seconds = 30.0
solver.parameters.num_search_workers = 8

status = solver.Solve(model)
if status not in (cp_model.OPTIMAL, cp_model.FEASIBLE):
    print("\n해를 찾지 못했습니다. (하드 규칙 조합 충돌 가능)")
    raise SystemExit

# =========================================================
# 12) 출력(이름 행 / 날짜 열) + 휴무 달력
# =========================================================
def get_row_shifts(i: int) -> list[str]:
    row = []
    for d in range(NUM_DAYS):
        assigned = "?"
        for s in shifts:
            if solver.Value(x[i, d, s]) == 1:
                assigned = SHIFT_NAMES[s]
                break
        row.append(assigned)
    return row

def print_off_calendar(year: int, month: int, shifts_list: list[str], title: str, off_symbol: str = "-", week_start: int = 0):
    cal = calendar.Calendar(firstweekday=week_start)
    print(f"\n[{title}] {year}-{month:02d} 휴무 달력 (휴무: [dd])")
    print("Mo Tu We Th Fr Sa Su" if week_start == 0 else "Su Mo Tu We Th Fr Sa")
    for week in cal.monthdayscalendar(year, month):
        cells = []
        for d in week:
            if d == 0:
                cells.append("  ")
            else:
                is_off = (shifts_list[d - 1] == off_symbol)
                cells.append(f"[{d:02d}]" if is_off else f" {d:02d}")
        print(" ".join(cells))

date_headers = [f"{(START_DATE + timedelta(days=d)).day:02d}({weekday_kor(START_DATE + timedelta(days=d))})"
                for d in range(NUM_DAYS)]

print("\n== 월간 근무표 (이름 행 / 날짜 열) ==")
print("\t".join(["이름"] + date_headers + ["D", "E", "N", "근무", "휴무", "OFF>=min", "OFF목표", "A패턴불일치"]))

rows_by_name = {}
for i, pid in enumerate(ALL_IDS):
    name = NAME_MAP[pid]
    row = get_row_shifts(i)
    rows_by_name[name] = row

    Dn = row.count("D")
    En = row.count("E")
    Nn = row.count("N")
    Wk = Dn + En + Nn
    Of = row.count("-")

    a_miss = "-"  # A만 표시
    if pid == "A":
        a_miss = str(sum(1 for d in range(NUM_DAYS) if (
            (d % 3 == 0 and row[d] != "E") or
            (d % 3 == 1 and row[d] != "-") or
            (d % 3 == 2 and row[d] != "D")
        )))

    print("\t".join([
        name,
        *row,
        str(Dn), str(En), str(Nn), str(Wk), str(Of),
        str(MIN_OFF_HARD),
        str(OFF_TARGET),
        str(a_miss),
    ]))

print("\n== 휴무 달력 출력 ==")
for name, row in rows_by_name.items():
    print_off_calendar(YEAR, MONTH, row, title=name, week_start=0)

print("\n== 날짜별 인원 체크(전체 기준) ==")
print("\t".join(["항목"] + date_headers))
D_line = ["D(전체)"]
E_line = ["E(전체)"]
N_line = ["N(전체)"]
OFF_line = ["-(전체)"]
for d in range(NUM_DAYS):
    d_cnt = sum(solver.Value(x[i, d, SHIFT_D]) for i in people_all)
    e_cnt = sum(solver.Value(x[i, d, SHIFT_E]) for i in people_all)
    n_cnt = sum(solver.Value(x[i, d, SHIFT_N]) for i in people_all)
    off_cnt = sum(solver.Value(x[i, d, SHIFT_OFF]) for i in people_all)

    D_line.append(str(d_cnt))
    E_line.append(str(e_cnt))
    N_line.append(str(n_cnt))
    OFF_line.append(str(off_cnt))

print("\t".join(D_line))
print("\t".join(E_line))
print("\t".join(N_line))
print("\t".join(OFF_line))

print("\n== 요약(체크) ==")
A_row = rows_by_name[NAME_MAP["A"]]
print("A(최영철) N 개수:", A_row.count("N"), "(반드시 0이어야 함)")
print("A(최영철) 1일(E) 선호 위반:", "예" if A_row[0] != "E" else "아니오")
print("비A N=6 위반 여부:")
for pid in STAFF_IDS:
    row = rows_by_name[NAME_MAP[pid]]
    print(pid, NAME_MAP[pid], "N=", row.count("N"))


== 빨간날(주말+공휴일+대체공휴일) ==  총 10일
== 휴무 기준 ==  하드 최소 9일, 목표 10일
== 빨간날 목록 ==
2026-03-01 3·1절
2026-03-02 대체 공휴일(3·1절)
2026-03-07 주말/추가
2026-03-08 주말/추가
2026-03-14 주말/추가
2026-03-15 주말/추가
2026-03-21 주말/추가
2026-03-22 주말/추가
2026-03-28 주말/추가
2026-03-29 주말/추가

== N 총량 체크 ==
sum(N_REQ_TOTAL) = 48 (목표 48)
N(1) 일수: 14 / N(2) 일수: 17

== 월간 근무표 (이름 행 / 날짜 열) ==
이름	01(일)	02(월)	03(화)	04(수)	05(목)	06(금)	07(토)	08(일)	09(월)	10(화)	11(수)	12(목)	13(금)	14(토)	15(일)	16(월)	17(화)	18(수)	19(목)	20(금)	21(토)	22(일)	23(월)	24(화)	25(수)	26(목)	27(금)	28(토)	29(일)	30(월)	31(화)	D	E	N	근무	휴무	OFF>=min	OFF목표	A패턴불일치
최영철	E	-	D	E	-	D	E	-	D	E	-	D	-	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	-	D	E	10	10	0	20	11	9	10	1
홍진우	-	-	D	-	D	-	D	D	-	N	-	N	N	-	E	N	-	E	E	E	E	-	D	-	D	D	N	-	E	E	N	7	7	6	20	11	9	10	-
김다영	-	D	E	-	D	D	-	E	N	-	E	-	D	D	D	-	N	-	N	N	-	-	E	N	-	E	-	E	-	N	-	6	6	6	18	13	9	10	-
강승민	D	D	-	D	-	N	-	-	E	E	E	N	-	E	-	D	D	D	D	-	E	E	-	E	-	N	N	-	-	N	N	7	7	6	20	11	9	10	-
문승환	E	E	-	N	-	E	N	-	-	N	N	-	E	E	-	E	-	E	N	N	-	-	D	D	D	-	D	D	D	-	D	7	7	6	20	11	9	10	-
라